# CIFAR-10 학습 코드

## Config

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import f1_score

In [6]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
BATCH_SIZE  = 128
LEARNING_RATE = 0.1
EPOCHS = 100
SAVE_PATH = "cifar10_model.pth"

print(f"device: {device}")

device: mps


In [7]:
stats = ((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))

In [14]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),       # 이미지를 랜덤하게 자름 (패딩 추가 후)
    transforms.RandomHorizontalFlip(),          # 좌우 반전
    transforms.ToTensor(),
    transforms.Normalize(*stats),               # 정규화
])

In [15]:
# 테스트 데이터는 정규화만 수행
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(*stats),
])

In [16]:
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)
train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)
test_loader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


In [8]:
# ----------------------------------------
# 3. 모델 정의 (Modified ResNet18)
# ----------------------------------------
# torchvision의 ResNet18을 가져와서 CIFAR-10용으로 수정합니다.
def get_cifar_resnet18():
    # 사전 학습된 가중치는 사용하지 않음 (From Scratch 학습)
    model = torchvision.models.resnet18(weights=None)

    # [수정 1] 첫 번째 Conv 레이어 수정
    # 원본: kernel_size=7, stride=2 (이미지를 너무 빨리 줄임)
    # 수정: kernel_size=3, stride=1 (32x32 해상도 유지)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

    # [수정 2] MaxPool 레이어 제거
    # 작은 이미지에서는 정보 손실이 큼
    model.maxpool = nn.Identity()

    # [수정 3] 마지막 Fully Connected Layer 수정 (출력 클래스 10개)
    model.fc = nn.Linear(model.fc.in_features, 10)

    return model

In [21]:
model = get_cifar_resnet18().to(device)

In [9]:
# ----------------------------------------
# 4. 최적화 도구 설정 (Optimizer & Scheduler)
# ----------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4)

# OneCycleLR: 학습률을 웜업했다가 서서히 낮추는 강력한 스케줄러
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS
)

NameError: name 'model' is not defined

In [23]:
# ----------------------------------------
# 5. 학습 루프 (Training Loop)
# ----------------------------------------
def train(epoch):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader, leave=True)
    for inputs, targets in loop:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        scheduler.step() # 배치마다 스케줄러 업데이트

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        loop.set_description(f"Epoch [{epoch}/{EPOCHS}]")
        loop.set_postfix(loss=train_loss/(total/BATCH_SIZE), acc=100.*correct/total)

In [24]:
def test(epoch):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    acc = 100. * correct / total
    print(f"Test Accuracy: {acc:.2f}%")
    return acc

In [25]:
# 실행
best_acc = 0
for epoch in range(1, EPOCHS + 1):
    train(epoch)
    acc = test(epoch)

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'best_resnet18_cifar10.pth')
        print(f"--> Best Model Saved! ({best_acc:.2f}%)")

Epoch [1/100]: 100%|██████████| 391/391 [01:00<00:00,  6.47it/s, acc=39.9, loss=1.62]


Test Accuracy: 50.47%
--> Best Model Saved! (50.47%)


Epoch [2/100]: 100%|██████████| 391/391 [00:58<00:00,  6.71it/s, acc=57.7, loss=1.18]


Test Accuracy: 56.81%
--> Best Model Saved! (56.81%)


Epoch [3/100]: 100%|██████████| 391/391 [00:57<00:00,  6.74it/s, acc=66.8, loss=0.939]


...

Test Accuracy: 95.08%


Epoch [99/100]: 100%|██████████| 391/391 [16:13<00:00,  2.49s/it, acc=99.9, loss=0.00363]   


Test Accuracy: 95.12%
--> Best Model Saved! (95.12%)


Epoch [100/100]: 100%|██████████| 391/391 [08:28<00:00,  1.30s/it, acc=99.9, loss=0.00421]   


Test Accuracy: 95.06%


In [33]:
print(f"최종 최고 정확도: {best_acc:.2f}%")

최종 최고 정확도: 95.12%


In [35]:
torch.save(model.state_dict(), SAVE_PATH)

In [10]:
# 1. 동일한 구조의 모델 객체 생성
loaded_model = get_cifar_resnet18()

# 2. 저장된 가중치 불러오기
PATH = "cifar10_model.pth"
loaded_model.load_state_dict(torch.load(PATH, map_location=device))

# 3. 장치 할당 및 평가 모드 전환
loaded_model.to(device)
loaded_model.eval()  # 추론 시에는 반드시 eval 모드로!
print("모델 로드 및 평가 모드 전환 완료")

모델 로드 및 평가 모드 전환 완료


/var/folders/hz/y31l2nc9675930r976t3k9sh0000gn/T/ipykernel_8783/594957255.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.load_state_dict(torch.load(PATH, m

In [12]:
from sklearn.metrics import f1_score, classification_report

def test(model):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)

            # MPS/GPU 데이터를 CPU로 옮기고 리스트에 추가
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    # F1-score 계산
    # 'macro': 모든 클래스의 F1을 평균냄 (클래스별 중요도 동일)
    # 'weighted': 클래스별 샘플 수에 따라 가중치 평균
    f1 = f1_score(all_targets, all_preds, average='macro')

    # 정확도 계산
    correct = sum([1 for p, t in zip(all_preds, all_targets) if p == t])
    acc = 100. * correct / len(all_targets)

    print(f"Test Accuracy: {acc:.2f}% | F1-score: {f1:.4f}")

    # 만약 각 클래스(비행기, 자동차 등)별 상세 지표를 보고 싶다면:
    # print(classification_report(all_targets, all_preds, target_names=trainset.classes))

    return acc, f1

In [17]:
# 실행
test(loaded_model)

Test Accuracy: 95.06% | F1-score: 0.9506


(95.06, 0.9505800430309863)